# Lead Research Agent — Playground

This notebook walks through each phase of the agent **one step at a time**.
You'll see exactly what Claude produces at each stage — the plan, the raw web findings, and the final lead profile.

**Flow:**
```
Company Name → [Planner] → ResearchPlan → [Executor] → Raw Findings → [Synthesizer] → LeadProfile
```

In [1]:
# Install all required packages into this notebook's Python environment.
# Run this cell once, then you're good — you won't need to run it again.
%pip install anthropic tavily-python httpx beautifulsoup4 pydantic python-dotenv rich

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
     ---------------------------------------- 0.0/109.4 kB ? eta -:--:--
     ---------------------------------------- 0.0/109.4 kB ? eta -:--:--
     --- ------------------------------------ 10.2/109.4 kB ? eta -:--:--
     ------------- ----------------------- 41.0/109.4 kB 653.6 kB/s eta 0:00:01
     --------------------------------- -- 102.4/109.4 kB 980.4 kB/s eta 0:00:01
     ------------------------------------ 109.4/109.4 kB 901.4 kB/s eta 0:00:00
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cach

---
## Step 0 — Setup

Keys are loaded automatically from the `.env` file in this folder.
Run the cell below to confirm everything is ready.

In [2]:
import os
import sys
from dotenv import load_dotenv

# Let Python find our local modules (models/, agent/, tools/)
sys.path.insert(0, '.')

# Load ANTHROPIC_API_KEY and TAVILY_API_KEY from .env
load_dotenv()

anthropic_ok = bool(os.environ.get("ANTHROPIC_API_KEY"))
tavily_ok    = bool(os.environ.get("TAVILY_API_KEY"))

print(f"ANTHROPIC_API_KEY : {'OK' if anthropic_ok else 'MISSING — check your .env'}")
print(f"TAVILY_API_KEY    : {'OK' if tavily_ok    else 'MISSING — check your .env'}")

ANTHROPIC_API_KEY : OK
TAVILY_API_KEY    : OK


---
## Step 1 — The Data Models

Before any AI runs, we define **what the data must look like**.

- `ResearchPlan` — what the Planner outputs (a list of research steps)
- `LeadProfile` — what the Synthesizer outputs (the final scored result)

Pydantic enforces these shapes at runtime. If Claude returns a wrong type, it crashes immediately
instead of silently corrupting data downstream.

The schemas below are passed **directly to Claude as tool definitions** — they tell Claude
exactly what fields to fill in and what types to use.

In [3]:
from models.plan import ResearchPlan, ToolType
from models.lead import LeadProfile
import json

print("=== ResearchPlan schema (what the Planner must output) ===")
print(json.dumps(ResearchPlan.model_json_schema(), indent=2))

=== ResearchPlan schema (what the Planner must output) ===
{
  "$defs": {
    "ResearchStep": {
      "properties": {
        "step_id": {
          "title": "Step Id",
          "type": "integer"
        },
        "description": {
          "description": "What this step is trying to find out",
          "title": "Description",
          "type": "string"
        },
        "tool": {
          "$ref": "#/$defs/ToolType",
          "description": "Which tool to use for this step"
        },
        "query": {
          "description": "The search query or URL to scrape",
          "title": "Query",
          "type": "string"
        }
      },
      "required": [
        "step_id",
        "description",
        "tool",
        "query"
      ],
      "title": "ResearchStep",
      "type": "object"
    },
    "ToolType": {
      "enum": [
        "search",
        "scrape"
      ],
      "title": "ToolType",
      "type": "string"
    }
  },
  "properties": {
    "company_name": {
      

In [4]:
print("=== LeadProfile schema (what the Synthesizer must output) ===")
print(json.dumps(LeadProfile.model_json_schema(), indent=2))

=== LeadProfile schema (what the Synthesizer must output) ===
{
  "$defs": {
    "KeyPerson": {
      "properties": {
        "name": {
          "title": "Name",
          "type": "string"
        },
        "title": {
          "title": "Title",
          "type": "string"
        }
      },
      "required": [
        "name",
        "title"
      ],
      "title": "KeyPerson",
      "type": "object"
    }
  },
  "properties": {
    "company_name": {
      "title": "Company Name",
      "type": "string"
    },
    "website": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Website"
    },
    "industry": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Industry"
    },
    "description": {
      "anyOf": [
        {
          "type": "string"
        },
        {
  

---
## Step 2 — The Planner

The Planner's job: take a company name and decide **what to research and how**.

Claude receives the `ResearchPlan` schema as a tool and is forced to call it (`tool_choice: any`).
This means it fills in structured fields instead of writing free-form text — much more reliable.

**Change the company name here:**

In [5]:
COMPANY = "OpenAI"  # <- change this to any company

In [7]:
from agent.planner import create_plan

print(f"Asking Claude to plan research for: {COMPANY}...\n")
plan = create_plan(COMPANY)

print(f"Claude generated {len(plan.steps)} research steps:\n")
for step in plan.steps:
    icon = "Search" if step.tool == ToolType.SEARCH else "Scrape"
    print(f"  Step {step.step_id} [{icon}]")
    print(f"    Goal : {step.description}")
    print(f"    Query: {step.query}")
    print()

Asking Claude to plan research for: OpenAI...

Claude generated 8 research steps:

  Step 1 [Search]
    Goal : Get a comprehensive overview of OpenAI — what they do, their mission, and who they serve
    Query: OpenAI company overview mission products and customers 2024

  Step 2 [Search]
    Goal : Assess company size, headcount, and growth trajectory
    Query: OpenAI number of employees headcount growth 2024

  Step 3 [Search]
    Goal : Identify latest funding rounds and valuation signals
    Query: OpenAI funding rounds valuation 2024 investment

  Step 4 [Search]
    Goal : Find key decision-makers including CEO, CTO, and engineering leadership
    Query: OpenAI CEO CTO Head of Engineering leadership team 2024

  Step 5 [Search]
    Goal : Research OpenAI's tech stack — programming languages, frameworks, and tools they use internally
    Query: OpenAI tech stack programming languages frameworks internal tools

  Step 6 [Scrape]
    Goal : Scrape OpenAI's careers page to identify

**Try this:** run the cell above twice on the same company — Claude is non-deterministic,
so the steps will differ slightly each time. The planner is where the agent's "thinking" happens.

---
## Step 3 — The Tools (no AI)

The tools are plain Python functions — no Claude, no randomness, no API cost.
Let's test them directly so you can see the raw data before Claude processes it.

In [8]:
from tools.web_search import search

query = f"{COMPANY} company overview funding headcount"
print(f"Searching: '{query}'\n")

results = search(query, max_results=3)

for i, r in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print(f"Title  : {r.title}")
    print(f"URL    : {r.url}")
    print(f"Snippet: {r.content[:300]}")
    print()

Searching: 'OpenAI company overview funding headcount'

--- Result 1 ---
Title  : OpenAI - 2026 Company Profile, Team, Funding & Competitors - Tracxn
URL    : https://tracxn.com/d/companies/openai/__kElhSG7uVGeFk1i71Co9-nwFtmtyMVT7f-YHMn4TFBg
Snippet: *   [About the company](https://tracxn.com/d/companies/openai/__kElhSG7uVGeFk1i71Co9-nwFtmtyMVT7f-YHMn4TFBg#about-the-company). *   [Funding and Investors](https://tracxn.com/d/companies/openai/__kElhSG7uVGeFk1i71Co9-nwFtmtyMVT7f-YHMn4TFBg#funding-and-investors). *   [Reports](https://tracxn.com/d/c

--- Result 2 ---
Title  : OpenAI - Crunchbase Company Profile & Funding
URL    : https://www.crunchbase.com/organization/openai
Snippet: • $810.9B Total Funding Amount • 1,380 Number of Investors. Track. Artificial Intelligence (AI) Companies With More Than 50 Employees (Top 10K) · 10,000 Number

--- Result 3 ---
Title  : OpenAI - Wikipedia
URL    : https://en.wikipedia.org/wiki/OpenAI
Snippet: [Jump to content](https://en.wikipedia.org/wiki/

In [9]:
from tools.web_scraper import scrape

# Grab one of the URLs from the search results above
url = results[0].url
print(f"Scraping: {url}\n")

content = scrape(url, max_chars=1200)
print(content)

Scraping: https://tracxn.com/d/companies/openai/__kElhSG7uVGeFk1i71Co9-nwFtmtyMVT7f-YHMn4TFBg

OpenAI - 2026 Company Profile, Team, Funding & Competitors - Tracxn
JavaScript is disabled in your browser. enable it to enjoy the full features of Tracxn.
Your browser was unable to load all of Tracxn resources. They may have been blocked by your firewall, proxy or browser configuration. Press
Ctrl+F5
or
Ctrl+Shift+R
to have your browser try again and if that doesn't work,
click here to retry
or mail us at
hi@tracxn.com
Internal Server Error
OpenAI - Company Profile
decacorn
Last updated:
May 16, 2026
Claim Profile
Suggest Edits
Linkedin
Twitter
Facebook
Email
Copy Url
Request page removal
Most viewed in 2019
OpenAI - About the company
OpenAI is a series  G company based in San Francisco (United States), founded in 2015
by
John Schulman
,
Wojciech Zaremba
,
Sam Altman
,
Ilya Sutskever
and
Elon Musk
.
It operates as a Developer of artificial intelligence models tools and research-driven platf

---
## Step 4 — The Executor

The Executor is just a `for` loop. It reads the plan and calls the right tool for each step.
No AI involved — this keeps it fast, cheap, and fully predictable.

The output is a dictionary: `{ topic description → raw text findings }`

> Takes ~20-40 seconds — it's making several web requests.

In [10]:
from agent.executor import execute_plan

print(f"Executing {len(plan.steps)} steps...\n")
raw_findings = execute_plan(plan)

print(f"Collected findings for {len(raw_findings)} topics:")
for topic in raw_findings:
    print(f"  - {topic}")

Executing 8 steps...

Collected findings for 8 topics:
  - Get a comprehensive overview of OpenAI — what they do, their mission, and who they serve
  - Assess company size, headcount, and growth trajectory
  - Identify latest funding rounds and valuation signals
  - Find key decision-makers including CEO, CTO, and engineering leadership
  - Research OpenAI's tech stack — programming languages, frameworks, and tools they use internally
  - Scrape OpenAI's careers page to identify active hiring areas and growth signals
  - Identify recent news, product launches, partnerships, or expansions
  - Scrape OpenAI's about page for additional company and mission details


In [11]:
# Inspect the raw text for any topic — change the index to explore others
topic = list(raw_findings.keys())[0]
print(f"=== Raw findings for: '{topic}' ===\n")
print(raw_findings[topic][:1500])

=== Raw findings for: 'Get a comprehensive overview of OpenAI — what they do, their mission, and who they serve' ===

[What are Mission Vision & Core Values of OpenAI Company? – businessmodelcanvastemplate.com](https://businessmodelcanvastemplate.com/blogs/mission/openai-mission)
# What Are OpenAI’s Mission, Vision, and Core Values? In the modern corporate landscape, mission and vision statements are far more than mere slogans; they represent the strategic bedrock upon which long-term success is built. This brief addresses the entity "Introduction" in the context of professional writing and content strategy: the opening section functions as a rhetorical and structural component that orients readers, establishes purpose, and sets the ethical and operational tone-linking product strategy like the OpenAI Canvas Business Model to broader governance goals while situating OpenAI among peers such as Anthropic, xAI, Mistral AI, Cohere, Hugging Face, Stability AI, Inflection AI, and Adept. ## M

**This is the messy middle.** Real web content is noisy, repetitive, and unstructured.
The Synthesizer's entire job is to make sense of this mess.

---
## Step 5 — The Synthesizer

Claude reads all the raw findings and produces a clean, typed `LeadProfile`.
Same trick as the Planner: we pass the `LeadProfile` schema as a tool and force Claude to call it.

> This is the most expensive Claude call — it reads everything collected above.

In [12]:
from agent.synthesizer import synthesize

print(f"Synthesizing profile for {COMPANY}...\n")
profile = synthesize(COMPANY, raw_findings)

print("Raw JSON output from Claude:")
print(profile.model_dump_json(indent=2))

Synthesizing profile for OpenAI...

Raw JSON output from Claude:
{
  "company_name": "OpenAI",
  "website": "https://openai.com",
  "industry": "Artificial Intelligence / Technology",
  "description": "OpenAI is an American artificial intelligence company focused on developing and deploying artificial general intelligence (AGI) for the benefit of humanity. It is the creator of ChatGPT, GPT models, DALL-E, and the Agents SDK, and serves over 1 million business customers with both consumer and enterprise AI products.",
  "employee_count_estimate": "~4,500 current (2026); targeting ~8,000 by end of 2026",
  "founding_year": 2015,
  "headquarters": "San Francisco, California, USA",
  "key_people": [
    {
      "name": "Sam Altman",
      "title": "CEO & Co-Founder"
    },
    {
      "name": "Greg Brockman",
      "title": "President & Co-Founder"
    },
    {
      "name": "Vijaye Raji",
      "title": "CTO of Applications"
    },
    {
      "name": "Jakub Pachocki",
      "title": "Chi

In [13]:
# Access individual fields
print(f"Company   : {profile.company_name}")
print(f"Industry  : {profile.industry}")
print(f"Size      : {profile.employee_count_estimate}")
print(f"Founded   : {profile.founding_year}")
print(f"HQ        : {profile.headquarters}")
print(f"Funding   : {profile.funding_info}")
print(f"Tech Stack: {', '.join(profile.tech_stack)}")
print(f"Hiring    : {profile.is_hiring}")
print(f"Score     : {profile.qualification_score}/10")
print(f"Reasoning : {profile.qualification_reasoning}")
print()
print("Key People:")
for p in profile.key_people:
    print(f"  - {p.name} ({p.title})")
print()
print("Talking Points for first outreach:")
for tp in profile.talking_points:
    print(f"  -> {tp}")
print()
print("Recent News:")
for news in profile.recent_news:
    print(f"  * {news}")

Company   : OpenAI
Industry  : Artificial Intelligence / Technology
Size      : ~4,500 current (2026); targeting ~8,000 by end of 2026
Founded   : 2015
HQ        : San Francisco, California, USA
Funding   : Most recent disclosed round: $6.6B Series E at $157B valuation (Oct 2024), led by Thrive Capital with Microsoft, Nvidia, and SoftBank participating. A subsequent round closed at a $300B valuation (as of April 2025), and a landmark $122B funding round at an $852B post-money valuation was reported more recently. Revenue projected at $11.6B in 2025, up from $3.7B in 2024.
Tech Stack: Python, JavaScript/TypeScript, PyTorch, NumPy, Next.js, React, Tailwind CSS, Shadcn/UI, Radix Themes, REST APIs, JSON, WebRTC, WebSocket
Hiring    : True
Score     : 3.0/10
Reasoning : OpenAI scores extremely high on company size, growth trajectory, budget signals, and leadership visibility — all hallmarks of a 10/10 target on those dimensions. However, the score is deliberately low (3/10) for a software d

In [16]:
from IPython.display import display, Markdown

def report(p) -> None:
    """Render a LeadProfile as a formatted markdown report in the notebook."""

    score = p.qualification_score
    if score >= 7:
        verdict, bar = "PURSUE", "🟢" * int(score) + "⬜" * (10 - int(score))
    elif score >= 4:
        verdict, bar = "MAYBE",  "🟡" * int(score) + "⬜" * (10 - int(score))
    else:
        verdict, bar = "PASS",   "🔴" * int(score) + "⬜" * (10 - int(score))

    key_people = "\n".join(f"- **{p.name}** — {p.title}" for p in p.key_people) or "_None found_"
    tech_stack = ", ".join(f"`{t}`" for t in p.tech_stack) or "_None found_"
    news       = "\n".join(f"- {n}" for n in p.recent_news) or "_None found_"
    points     = "\n".join(f"- {t}" for t in p.talking_points) or "_None found_"

    md = f"""
---
# {p.company_name}

> {p.description or "_No description found_"}

## At a Glance

| Field | Value |
|---|---|
| Industry | {p.industry or "—"} |
| Size | {p.employee_count_estimate or "—"} |
| Founded | {p.founding_year or "—"} |
| Headquarters | {p.headquarters or "—"} |
| Website | {p.website or "—"} |
| Funding | {p.funding_info or "—"} |
| Actively Hiring | {"Yes" if p.is_hiring else "No" if p.is_hiring is False else "—"} |

## Key People
{key_people}

## Tech Stack
{tech_stack}

## Recent News
{news}

## Qualification

**Score: {score}/10 — {verdict}**

{bar}

_{p.qualification_reasoning}_

## Talking Points for First Outreach
{points}

---
"""
    display(Markdown(md))


# Render the profile from Step 5
report(profile)


---
# OpenAI

> OpenAI is an American artificial intelligence company focused on developing and deploying artificial general intelligence (AGI) for the benefit of humanity. It is the creator of ChatGPT, GPT models, DALL-E, and the Agents SDK, and serves over 1 million business customers with both consumer and enterprise AI products.

## At a Glance

| Field | Value |
|---|---|
| Industry | Artificial Intelligence / Technology |
| Size | ~4,500 current (2026); targeting ~8,000 by end of 2026 |
| Founded | 2015 |
| Headquarters | San Francisco, California, USA |
| Website | https://openai.com |
| Funding | Most recent disclosed round: $6.6B Series E at $157B valuation (Oct 2024), led by Thrive Capital with Microsoft, Nvidia, and SoftBank participating. A subsequent round closed at a $300B valuation (as of April 2025), and a landmark $122B funding round at an $852B post-money valuation was reported more recently. Revenue projected at $11.6B in 2025, up from $3.7B in 2024. |
| Actively Hiring | Yes |

## Key People
- **Sam Altman** — CEO & Co-Founder
- **Greg Brockman** — President & Co-Founder
- **Vijaye Raji** — CTO of Applications
- **Jakub Pachocki** — Chief Scientist
- **Brad Lightcap** — Chief Operating Officer
- **Sarah Friar** — Chief Financial Officer
- **Kevin Weil** — Chief Product Officer
- **Srinivas Narayanan** — VP of Engineering
- **Mark Chen** — SVP of Research
- **Jason Kwon** — Chief Strategy Officer

## Tech Stack
`Python`, `JavaScript/TypeScript`, `PyTorch`, `NumPy`, `Next.js`, `React`, `Tailwind CSS`, `Shadcn/UI`, `Radix Themes`, `REST APIs`, `JSON`, `WebRTC`, `WebSocket`

## Recent News
- OpenAI plans to nearly double headcount from ~4,500 to ~8,000 employees by end of 2026, primarily in engineering, product, research, and sales (March 2026).
- OpenAI closed a $122B funding round at an $852B post-money valuation, the largest in company history.
- Amazon announced plans to invest up to $50B in OpenAI as part of a strategic partnership (Feb 2026).
- OpenAI deepened partnerships with major consulting firms (Bain & Co, and others) to distribute AI tools to enterprise clients.
- OpenAI signed a multi-year strategic deal with AMD to deploy 6 gigawatts of next-generation AMD Instinct infrastructure.
- OpenAI secured a contract with the US Department of Defense to deploy AI models.
- ChatGPT message volume grew 8x year-over-year; API reasoning token consumption per org grew 320x YoY.
- OpenAI's website averaged 663.6 million monthly visits between April–June 2025.

## Qualification

**Score: 3.0/10 — PASS**

🔴🔴🔴⬜⬜⬜⬜⬜⬜⬜

_OpenAI scores extremely high on company size, growth trajectory, budget signals, and leadership visibility — all hallmarks of a 10/10 target on those dimensions. However, the score is deliberately low (3/10) for a software development agency because OpenAI is itself one of the world's most advanced AI and software development organizations. With 4,500+ engineers, researchers, and product developers — and a plan to grow to 8,000 — they build their own technology in-house at the cutting edge. They are unlikely to outsource software development to an agency, and are far more likely to be a competitor or a tooling provider than a buyer of dev services. The relevance to a software development agency as a potential client is very low._

## Talking Points for First Outreach
- OpenAI is aggressively hiring across engineering and product — if internal capacity is stretched during this doubling of headcount, there could be a narrow window to offer specialized or niche dev capacity (e.g., legacy integrations, enterprise client customization work).
- OpenAI is expanding into enterprise via consulting firm partnerships (Bain, etc.) — a dev agency with enterprise integration expertise could potentially enter as a downstream implementation partner rather than a direct vendor.
- The push into government contracts (DoD) may create compliance-heavy software needs (FedRAMP, etc.) where specialized agencies play a role.
- Their developer platform (Agents SDK, Realtime API, etc.) is growing fast — a dev agency that builds on top of OpenAI's APIs and can demonstrate deep platform expertise may find an 'ecosystem partner' angle rather than a vendor pitch.

---


---
## Step 6 — Full Pipeline

Now that you've seen each phase individually, here's the whole thing in one function.
This is exactly what `main.py` does for each company in your list.

In [14]:
def research_company_full(company_name: str) -> LeadProfile:
    print(f"[1/3] Planning for '{company_name}'...")
    plan = create_plan(company_name)
    print(f"      {len(plan.steps)} steps planned")

    print(f"[2/3] Executing research steps...")
    findings = execute_plan(plan)
    print(f"      {len(findings)} topics collected")

    print(f"[3/3] Synthesizing lead profile...")
    profile = synthesize(company_name, findings)
    print(f"      Score: {profile.qualification_score}/10")

    return profile


result = research_company_full("Linear")

print()
print(f"LEAD     : {result.company_name}")
print(f"SCORE    : {result.qualification_score}/10")
print(f"VERDICT  : {result.qualification_reasoning}")

[1/3] Planning for 'Linear'...
      7 steps planned
[2/3] Executing research steps...
      7 topics collected
[3/3] Synthesizing lead profile...
      Score: 4.0/10

LEAD     : Linear
SCORE    : 4.0/10
VERDICT  : Linear is a high-growth, well-funded SaaS company ($134M raised, $1.25B valuation, $100M ARR) with strong budget signals and visible decision-makers (named CTO and CEO are public and reachable). However, it scores LOW as a target for a software development agency for several key reasons: (1) Linear IS a software company — they build their own product with an internal engineering team that is deeply proud of its craft and explicitly values keeping the team small and doing more with less; (2) their hiring philosophy ("keep the team small, do more with less") and culture of internal craftsmanship makes outsourcing unlikely; (3) they are not a buyer of external dev services — they ARE the dev services. There is no realistic angle for selling software development agency services 

---
## Experiment Zone

Ideas to build intuition:

1. **Run the Planner twice** on the same company — steps change each time (non-determinism)
2. **Try a tiny unknown company** — watch how the score drops and null fields appear
3. **Change `[0]` to `[2]`** in the raw findings inspector — compare different topics
4. **Edit the system prompt** in `agent/synthesizer.py` to score for a different buyer profile and see how scores shift
5. **Add a new field** to `LeadProfile` in `models/lead.py` (e.g. `competitors`) and rerun — Claude will fill it automatically

In [17]:
# Free sandbox — try any company here
my_company = "NOW GmbH"
my_profile = research_company_full(my_company)
report(my_profile)

[1/3] Planning for 'NOW GmbH'...
      7 steps planned
[2/3] Executing research steps...
      7 topics collected
[3/3] Synthesizing lead profile...
      Score: 3.0/10



---
# NOW GmbH

> NOW GmbH (Nationale Organisation für den Wandel in der Mobilität) is a federally owned German company that coordinates and supports strategic projects in climate-friendly mobility and sustainable energy supply. It implements and manages government funding programs covering electromobility, charging infrastructure, hydrogen and fuel cell technology, and alternative fuels. It acts as an interface between politics, industry, and research, and recently rebranded in 2026 by removing "Hydrogen" from its name to reflect a broader mobility mandate.

## At a Glance

| Field | Value |
|---|---|
| Industry | Government / Public Sector – Climate-Friendly Mobility & Sustainable Energy |
| Size | 51–200 |
| Founded | — |
| Headquarters | Berlin, Germany |
| Website | https://www.now-gmbh.de/en/ |
| Funding | — |
| Actively Hiring | Yes |

## Key People
_None found_

## Tech Stack
_None found_

## Recent News
- NOW GmbH rebranded in early 2026, removing 'Hydrogen' from its name to reflect a broader mandate covering all climate-friendly mobility technologies.
- The Federal Ministry of Transport (BMV) is funding one billion euros for the development of charging infrastructure for heavy-duty vehicles, coordinated by NOW GmbH.
- NOW GmbH is participating in IAA Transportation Messe 2026 (Sept 15–20, 2026) and Power2Drive Messe 2026 (June 23–25, 2026).
- NOW GmbH published a new quick guide in May 2026 to support logistics companies in building truck charging infrastructure at depots.

## Qualification

**Score: 3.0/10 — PASS**

🔴🔴🔴⬜⬜⬜⬜⬜⬜⬜

_NOW GmbH is a federally owned German government agency (not a private company) focused on coordinating climate-friendly mobility and sustainable energy funding programs. As a public-sector body with 51–200 employees, it has limited budget autonomy typical of commercial enterprises, and procurement decisions are subject to government frameworks (likely requiring formal tendering processes). No named CTO or Head of Engineering was identified, making direct outreach to a technical decision-maker difficult. The organization's core mission is policy coordination and funding administration rather than building commercial software products, which significantly reduces the likelihood they would engage a software development agency for ongoing tech services. The rebrand news and active event calendar show organizational activity, but there are no signals of tech investment, funding rounds, or engineering hiring that would indicate a near-term need for external software development services._

## Talking Points for First Outreach
- NOW GmbH manages complex multi-stakeholder funding programs — a custom digital portal or case management tool could streamline applicant tracking and reporting.
- The organization's rebranding and expanded mandate in 2026 may create a need for updated digital presence, web platforms, or data visualization tools.
- As a knowledge hub bridging policy, industry, and research, NOW GmbH could benefit from modern knowledge management or event/webinar platforms to support its growing seminar and workshop calendar.

---
